# Imports

## Standard Library

In [1]:
import os          # file paths, directory management
import sys         # system-level operations (rarely used, but useful for debugging)
import time        # timing execution (performance measurement)
import gc          # garbage collection (manual memory cleanup)

## Data Handling & Numerical Computation

In [2]:
import numpy as np         # numerical operations, arrays, embeddings
import pandas as pd        # dataset loading and manipulation (DataFrames)

## System Monitoring & External Communication

In [3]:
import psutil              # monitor RAM/CPU usage (GreenAI metrics)
import requests            # send notifications (by NTFY)

## Image Processing & Computer Vision

In [4]:
from PIL import Image      # image loading
import cv2                 # OpenCV (advanced image processing if needed)

## Progress & Visualization

In [5]:
from tqdm.notebook import tqdm   # progress bars

import matplotlib.pyplot as plt  # plotting results
import seaborn as sns            # statistical visualizations

## Statistics / Evaluation

In [6]:
from scipy.stats import spearmanr   # rank correlation (similarity preservation)

## PyTorch (Core Deep Learning Framework)

In [7]:
import torch
import torch.nn as nn              # neural network layers

## TorchVision (Pretrained Vision Models & Transforms)

In [8]:
from torchvision import models, transforms

## Transformers (HuggingFace Models)

In [9]:
from transformers import (
    # Vision Transformers
    ViTModel, ViTImageProcessor,
    DeiTModel, DeiTImageProcessor,

    # Generic auto models (flexible loading)
    AutoImageProcessor, AutoModel,

    # CLIP (multimodal)
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,

    # Text models
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

# Configuration

## Dataset Selection

In [10]:
# Available datasets: Flickr8k, Flickr30k, ConceptualCaptions
CURRENT_DATASET = "Flickr8k" 
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]

assert CURRENT_DATASET in ALL_DATASETS, f"{CURRENT_DATASET} is not a valid dataset"


## Directory Architecture

In [11]:

BASE_DIR = os.path.join(os.getcwd(), 'TFE_Data')

DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
RAW_DIR = os.path.join(BASE_DIR, 'Features_RAW', CURRENT_DATASET)
RESULTS_DIR = os.path.join(BASE_DIR, 'Results', CURRENT_DATASET)

# Create core directories
for d in [DATASETS_DIR, RAW_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)


## Saliency Maps Directory (per dataset)

In [12]:
def get_saliency_dir(dataset_name):
    path = os.path.join(BASE_DIR, f"SaliencyMaps_{dataset_name}")
    os.makedirs(path, exist_ok=True)
    return path

## Device Configuration

In [13]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


## Notification Setup

In [14]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    NTFY_TOPIC = "aysel_tfe_server_9988"
    try:
        requests.post(
            f"https://ntfy.sh/{'aysel_tfe_server_9988'}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")



# Data Loading

In [15]:
print(f"Loading dataset: {CURRENT_DATASET}...")

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
if not os.path.exists(df_path):
    raise FileNotFoundError(f"Dataset file not found at {df_path}. Please run data_builder.ipynb first.")

df = pd.read_pickle(df_path)

# Extract inputs for models
IMAGE_PATHS = df['image_path'].tolist()

CAPTIONS_LIST = df['captions'].tolist()
assert len(IMAGE_PATHS) == len(CAPTIONS_LIST) # sanity check: each image should have a list of captions
print(f"Ready. Loaded {len(IMAGE_PATHS)} images.")
print(f"Each image has {len(CAPTIONS_LIST[0])} captions (example).")

texts_for_xai = [caps[0] for caps in CAPTIONS_LIST] # For XAI, we will use the first caption of each image to keep it simple.
print(f"Ready. Loaded {len(IMAGE_PATHS)} images and {len(texts_for_xai)} captions into memory.")

Loading dataset: Flickr8k...
Ready. Loaded 8091 images.
Each image has 5 captions (example).
Ready. Loaded 8091 images and 8091 captions into memory.


# Utility Functions

## GreenAI Metrics

In [16]:
def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [17]:
def get_size_in_mb(obj):
    """Returns the size of a numpy array in MB."""
    if isinstance(obj, np.ndarray):
        return obj.nbytes / (1024 * 1024)
    else:
        return sys.getsizeof(obj) / (1024 * 1024)

In [18]:
def execute_and_save(modality, model_name, extract_func, data, device):
    """Measures metrics, extracts features, and saves results."""
    
    print(f"\nProcessing {modality.upper()} with model: {model_name}")
    
    start_time = time.time()
    mem_before = measure_memory()
    
    try:
        with torch.no_grad():
            features = extract_func(data, device)
    except Exception as e:
        print(f"Error with {model_name}: {e}")
        return None
    
    if features is None:
        print(f"Skipping {model_name} due to extraction failure.")
        return None
    
    exec_time = time.time() - start_time
    mem_used = max(0, measure_memory() - mem_before)
    
    save_path = os.path.join(
        RAW_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy"
    )
    np.save(save_path, features)
    
    print(f"Saved RAW features to: {save_path}")
    
    original_dim = features.shape[1] if features.ndim > 1 else 1
    original_size = get_size_in_mb(features)
    
    # Memory cleanup
    del features
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {
        "Modality": modality,
        "Model": model_name,
        "Original_Dim": original_dim,
        "Exec_Time_s": exec_time,
        "Memory_Used_MB": mem_used,
        "Original_Size_MB": original_size
    }

## XAI Saliency Maps

### Vision XAI

In [19]:
def save_saliency_and_overlay(model_name, activations, img_idx, img_path, dataset):
    """
    Captures pertinent zones using Norm-based weighting.
    Fixes the token dimension mismatch by explicitly removing the [CLS] token.
    """
    act = activations.detach().cpu()
    
    # --- CNN Logic (4D Tensors) ---
    if len(act.shape) == 4:
        weights = torch.norm(act, dim=1, keepdim=True)
        heatmap = torch.sum(act * weights, dim=1).squeeze().numpy()
        
    # --- Transformer Logic (3D Tensors) ---
    elif len(act.shape) == 3:
        # act shape: [1, 197, 768] (for DeiT/ViT)
        # We must remove the [CLS] token at index 0
        tokens = act[:, 1:, :]  # remove CLS

        # DeiT distilled → parfois 198 tokens
        if tokens.shape[1] == 197:
            tokens = tokens[:, :-1, :]
        
        # Calculate importance weights for the 196 patches
        weights = torch.norm(tokens, dim=-1, keepdim=True) # Shape: [1, 196, 1]
        
        # Apply weighting and sum feature dimensions -> Shape: [1, 196]
        weighted_sum = torch.sum(tokens * weights, dim=-1)
        
        # Calculate grid size (sqrt of 196 is 14)
        num_patches = tokens.shape[1]
        side = int(np.sqrt(num_patches)) 
        
        try:
            # Reshape exactly 196 elements into [14, 14]
            heatmap = weighted_sum.view(side, side).numpy()
        except RuntimeError:
            # Fallback if patch count is unexpected
            return
            
    else: return

    # Normalize heatmap 0-1
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
    heatmap_res = cv2.resize(heatmap, (224, 224))
    
    # Save Raw Saliency Map (.npy)
    saliency_dir = os.path.join(BASE_DIR, f"SaliencyMaps_{dataset}")
    os.makedirs(saliency_dir, exist_ok=True)
    np.save(os.path.join(saliency_dir, f"Saliency_{model_name}_{img_idx}.npy"), heatmap_res)
    
    # Save Visual Overlay 
    img_bgr = cv2.imread(img_path)
    img_bgr = cv2.resize(img_bgr, (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)
    cv2.imwrite(os.path.join(RESULTS_DIR, f"Overlay_{dataset}_{model_name}_{img_idx}.jpg"), overlay)

### Text XAI

In [20]:
def save_text_saliency(tokens, scores, model_name, idx, dataset_name):
    save_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    os.makedirs(save_dir, exist_ok=True)

    path = os.path.join(
        save_dir,
        f"SaliencyText_{model_name}_{idx}.npy"
    )

    np.save(path, {"tokens": tokens, "scores": scores})

# Unimodal Models

## CBIR: Vision Feature Extractions

### Image Dataset

In [21]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))  # fallback black image

        if self.transform:
            try:
                image = self.transform(image)
            except Exception:
                image = torch.zeros((3, 224, 224))  # fallback tensor if transform fails

        return image

### Feature Extraction

In [22]:
def vision_extract_features(model, model_name, dataloader, device, target_layer,
                                   paths, dataset_name, save_saliency=True):
    """
    Generic function to extract features from a model and optionally save saliency maps.
    Handles CNNs and Transformers automatically.
    """
    model.eval()
    features_list = []
    activations = {}
    img_idx = 0

    # Hook function for capturing intermediate activations
    def hook_fn(module, input, output):
        if hasattr(output, "last_hidden_state"):
            activations['value'] = output.last_hidden_state
        elif isinstance(output, tuple):
            activations['value'] = output[0]
        else:
            activations['value'] = output

    hook = None
    if target_layer is not None:
        hook = target_layer.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"{model_name.lower()}"):
            batch = batch.to(device)

            try:
                output = model(batch)
            except Exception as e:
                print(f"[ERROR] Model forward pass failed for {model_name}: {e}")
                continue

            # --- Feature Extraction ---
            if hasattr(output, 'last_hidden_state'):
                feat = output.last_hidden_state[:, 0, :]
            elif isinstance(output, torch.Tensor):
                # CNN case: global average pool if 4D tensor
                if output.ndim == 4:
                    feat = torch.nn.functional.adaptive_avg_pool2d(output, 1).view(output.size(0), -1)
                else:
                    feat = output
            else:
                feat = output[0]
            features_list.append(feat.cpu().numpy())

            # --- Saliency Map Saving ---
            if save_saliency:
                batch_acts = activations.get('value')
                if batch_acts is None:
                    print(f"[WARNING] No activations captured for {model_name}")
                else:
                    for i in range(batch_acts.size(0)):
                        save_saliency_and_overlay(
                            model_name.lower(),
                            batch_acts[i:i+1],
                            img_idx,
                            paths[img_idx],
                            dataset_name
                        )
                        img_idx += 1
                    activations['value'] = None

    if hook:
        hook.remove()

    return np.vstack(features_list)

#### CNN Embeddings

In [23]:
def get_resnet50_embeddings(image_paths, device, dataset_name):
    try:
        model = models.resnet50(weights="DEFAULT").to(device)
        target_layer = model.layer4[-1]
        model.fc = nn.Identity()  # Remove classifier

        transform = models.ResNet50_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "resnet50", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] ResNet50 embedding failed: {e}")
        return None

In [24]:
def get_mobilenet_v3_embeddings(image_paths, device, dataset_name):
    try:
        model = models.mobilenet_v3_large(weights="DEFAULT").to(device)
        target_layer = model.features[-1]
        model.classifier = nn.Identity()  # Remove classifier

        transform = models.MobileNet_V3_Large_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "mobilenet_v3", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] MobileNetV3 embedding failed: {e}")
        return None

#### Transformer Embeddings

In [25]:
def get_vit_embeddings(image_paths, device, dataset_name):
    try:
        processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
        model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
        target_layer = model.encoder.layer[-1]

        def collate_fn(batch):
            return torch.stack([processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0) for img in batch])

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        return vision_extract_features(model, "vit", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] ViT embedding failed: {e}")
        return None

In [26]:
def get_deit_embeddings(image_paths, device, dataset_name):
    try:
        processor = DeiTImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
        model = DeiTModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device)
        target_layer = model.encoder.layer[-1].output

        def collate_fn(batch):
            return torch.stack([processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0) for img in batch])

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        return vision_extract_features(model, "deit", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] DeiT embedding failed: {e}")
        return None

In [27]:

def get_clip_vision_embeddings(image_paths, device, dataset_name):
    try:
        processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        model = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
        target_layer = model.vision_model.encoder.layers[-1]

        def collate_fn(batch):
            return torch.stack([processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0) for img in batch])

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        return vision_extract_features(model, "clip_vision", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] CLIP Vision embedding failed: {e}")
        return None

### Vision Pipeline Registry

In [28]:
vision_pipeline = {
    "resnet50": get_resnet50_embeddings,
    "mobilenet_v3": get_mobilenet_v3_embeddings,
    "vit": get_vit_embeddings,
    "deit": get_deit_embeddings,
    "clip_vision": get_clip_vision_embeddings
}

## T2T: Text Feature Extractions


### Text Dataset

In [29]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

### Multiple Captions Encoder

In [30]:
def encode_multiple_captions(captions, model, tokenizer, device):
    """
    Encodes multiple captions for the same image and averages the embeddings.
    """
    embeddings = []
    for cap in captions:
        inputs = tokenizer(
            cap, return_tensors="pt", truncation=True, padding=True
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
            embeddings.append(emb.cpu().numpy())
    return np.mean(embeddings, axis=0)

## Text Feature Extraction

In [31]:
def text_extract_features(model, tokenizer, dataloader, device, dataset_name, model_name, feature_type='mean_pool', save_xai=True):
    """
    Extract embeddings and optionally save token-level saliency maps.
    """
    model.eval()
    features_list = []

    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    os.makedirs(saliency_dir, exist_ok=True)

    xai_index = 0
    for batch in tqdm(dataloader, desc=f"{model_name} Text Extraction"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        # Feature extraction
        if feature_type == 'mean_pool':
            mask = batch['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_emb = torch.sum(outputs.last_hidden_state * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            batch_features = (sum_emb / sum_mask).cpu().numpy()
        else:
            # CLS token
            batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        features_list.append(batch_features)

        # Token-level XAI
        if save_xai:
            for i in range(outputs.last_hidden_state.size(0)):
                tokens = tokenizer.convert_ids_to_tokens(batch['input_ids'][i])
                scores = outputs.last_hidden_state[i].norm(dim=-1).cpu().numpy()
                saliency_path = os.path.join(saliency_dir, f"SaliencyText_{model_name}_{xai_index}.npy")
                np.save(saliency_path, {'tokens': tokens, 'scores': scores})
                xai_index += 1

    return np.vstack(features_list)

## Specialized RNN for text

In [32]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.rnn(embedded)
        return hidden[-1], output  # last hidden for embedding, full seq for XAI

# ----------------------------
# RNN Embedding + XAI
# ----------------------------
def get_rnn_embeddings(texts, device, dataset_name):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = SimpleRNN(tokenizer.vocab_size).to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)

    features_list = []
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    os.makedirs(saliency_dir, exist_ok=True)

    model.eval()
    for idx, batch in enumerate(tqdm(dataloader, desc="RNN XAI")):
        input_ids = batch['input_ids'].to(device)
        with torch.no_grad():
            last_hidden, full_seq = model(input_ids)
            features_list.append(last_hidden.cpu().numpy())

        # Token-level saliency: magnitude of LSTM output
        for b in range(full_seq.size(0)):
            tokens = tokenizer.convert_ids_to_tokens(input_ids[b])
            scores = full_seq[b].norm(dim=-1).cpu().numpy()
            saliency_path = os.path.join(saliency_dir, f"SaliencyText_rnn_{idx*32 + b}.npy")
            np.save(saliency_path, {'tokens': tokens, 'scores': scores})

    return np.vstack(features_list)

### Text Models

In [33]:
def get_bert_embeddings(texts, device, dataset_name):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "bert")


In [34]:

def get_roberta_embeddings(texts, device, dataset_name):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "roberta")


In [35]:

def get_gpt2_embeddings(texts, device, dataset_name):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "gpt2")


In [36]:
def get_clip_text_embeddings(texts, device, dataset_name=None, max_length=32):
    from transformers import CLIPTextModel, CLIPTokenizer
    import torch
    import numpy as np
    from tqdm import tqdm

    tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

    features = []
    model.eval()
    batch_size = 32

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting CLIP Text"):
        batch_texts = texts[i:i+batch_size]
        # Make sure max_length is an int
        encoding = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=int(max_length)
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
            features.append(batch_features)

    return np.vstack(features)

## Text Pipeline Registry

In [37]:
text_pipeline = {
    "bert": get_bert_embeddings,
    "roberta": get_roberta_embeddings,
    "gpt2": get_gpt2_embeddings,
    "clip_text": get_clip_text_embeddings,
    "rnn": get_rnn_embeddings
}

## Execution

In [38]:
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]
all_metrics = []

for ds_name in ALL_DATASETS:
    print(f"\n{'='*25}\nProcessing Dataset: {ds_name}\n{'='*25}")

    # --- Load Dataset ---
    df_path = os.path.join(DATASETS_DIR, f"df_{ds_name}.pkl")
    if not os.path.exists(df_path):
        print(f"Skipping {ds_name}: Metadata not found.")
        continue
    df = pd.read_pickle(df_path)

    IMAGE_PATHS = df['image_path'].tolist()
    # Flatten all captions for text pipeline
    CAPTIONS_FLAT = [cap for caps in df['captions'] for cap in caps]

    send_ntfy_notification(f"Starting Indexation for {ds_name}", "TFE Pipeline")
    """
    # --- VISION PIPELINE ---
    print(f"\n{'='*20}\nVISION PIPELINE: {ds_name}\n{'='*20}")
    for model_name, func in vision_pipeline.items():
        print(f"\n--- Extracting {model_name.upper()} embeddings and saliency ---")
        feats = func(IMAGE_PATHS, device, ds_name)
        if feats is not None:
            all_metrics.append({
                "Dataset": ds_name,
                "Model": model_name,
                "Type": "vision",
                "Features": feats
            })
    """
    # --- TEXT PIPELINE ---
    print(f"\n{'='*20}\nTEXT PIPELINE: {ds_name}\n{'='*20}")
    # Keep track of per-image embeddings after averaging all captions
    for model_name, func in text_pipeline.items():
        print(f"\n--- Extracting {model_name.upper()} embeddings and saliency ---")
        # Extract embeddings for all captions
        caption_feats = func(CAPTIONS_FLAT, device, ds_name)  # Returns flattened list
        # Reshape/aggregate to per-image (average across captions)
        per_image_feats = []
        idx = 0
        for caps in df['captions']:
            num_caps = len(caps)
            # Average embeddings for this image
            img_feat = np.mean(caption_feats[idx:idx+num_caps], axis=0)
            per_image_feats.append(img_feat)
            idx += num_caps

        per_image_feats = np.vstack(per_image_feats)
        all_metrics.append({
            "Dataset": ds_name,
            "Model": model_name,
            "Type": "text",
            "Features": per_image_feats
        })

    send_ntfy_notification(f"Completed {ds_name}", "TFE Pipeline Success")

# --- Save Global Results ---
df_final = pd.DataFrame(all_metrics)
df_final.to_pickle(os.path.join(RESULTS_DIR, "global_unimodal_metrics.pkl"))
print("All datasets and models processed successfully.")


Processing Dataset: Flickr8k



TEXT PIPELINE: Flickr8k

--- Extracting BERT embeddings and saliency ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bert Text Extraction:   0%|          | 0/1265 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.


--- Extracting RNN embeddings and saliency ---


RNN XAI:   0%|          | 0/1563 [00:00<?, ?it/s]

All datasets and models processed successfully.


In [39]:
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]
# --- Global Execution Loop ---
all_metrics = []

for ds_name in ALL_DATASETS:
    # 1. Dataset Loading
    df_path = os.path.join(DATASETS_DIR, f"df_{ds_name}.pkl")
    if not os.path.exists(df_path):
        print(f"Skipping {ds_name}: Metadata not found."); continue
        
    df = pd.read_pickle(df_path)
    IMAGE_PATHS = df['image_path'].tolist()
    texts_for_xai = [c[0] if isinstance(c, list) else c for c in df['captions']]

    send_ntfy_notification(f"Starting Indexation for {ds_name}", "TFE Pipeline")

    # 2. Vision Pipeline Execution
    print(f"\n{'='*20}\nVISION: {ds_name}\n{'='*20}")
    # Note: We use a lambda to pass the dataset name into the wrapper
    vision_pipeline = {
        "resnet50": lambda p, d: get_resnet50_embeddings(p, d, ds_name),
        "mobilenet_v3": lambda p, d: get_mobilenet_v3_embeddings(p, d, ds_name),
        "vit": lambda p, d: get_vit_embeddings(p, d, ds_name),
        "deit": lambda p, d: get_deit_embeddings(p, d, ds_name),
        "clip_vision": lambda p, d: get_clip_vision_embeddings(p, d, ds_name)
    }

    for name, func in vision_pipeline.items():
        metrics = execute_and_save("vision", name, func, IMAGE_PATHS, device)
        all_metrics.append({**metrics, "Dataset": ds_name})
    
    # 3. Text Pipeline Execution
    print(f"\n{'='*20}\nTEXT: {ds_name}\n{'='*20}")
    # Text models do not require the 'ds_name' inside the wrapper
    text_pipeline = {
        "rnn": get_rnn_embeddings, "bert": get_bert_embeddings,
        "roberta": get_roberta_embeddings, "gpt2": get_gpt2_embeddings,
        "clip_text": get_clip_text_embeddings
    }

    for name, func in text_pipeline.items():
        metrics = execute_and_save("text", name, func, texts_for_xai, device)
        all_metrics.append({**metrics, "Dataset": ds_name})

    send_ntfy_notification(f"Completed {ds_name}", "TFE Pipeline Success")

# --- Save Global Results ---
df_final = pd.DataFrame(all_metrics)
df_final.to_pickle(os.path.join(RESULTS_DIR, "global_unimodal_metrics.pkl"))
print("All datasets and models processed successfully.")


VISION: Flickr8k

Processing VISION with model: resnet50


resnet50:   0%|          | 0/253 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/Flickr8k/X_vision_resnet50_raw_Flickr8k.npy

Processing VISION with model: mobilenet_v3


mobilenet_v3:   0%|          | 0/253 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/Flickr8k/X_vision_mobilenet_v3_raw_Flickr8k.npy

Processing VISION with model: vit


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias 

clip_vision:   0%|          | 0/253 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/Flickr8k/X_vision_clip_vision_raw_Flickr8k.npy

TEXT: Flickr8k

Processing TEXT with model: rnn
Error with rnn: get_rnn_embeddings() missing 1 required positional argument: 'dataset_name'


TypeError: 'NoneType' object is not a mapping

In [ ]:
import matplotlib.pyplot as plt

def validate_saliency_maps(dataset_name, num_images=3):
    """
    Displays a comparison grid: Original Image vs. 5 Model Saliency Maps.
    """
    vision_models = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
    saliency_dir = os.path.join(BASE_DIR, f"SaliencyMaps_{dataset_name}")
    
    # Load dataset metadata to get original image paths
    df_path = os.path.join(DATASETS_DIR, f"df_{dataset_name}.pkl")
    df_temp = pd.read_pickle(df_path)
    
    for idx in range(num_images):
        img_path = df_temp.iloc[idx]['image_path']
        img_rgb = Image.open(img_path).convert('RGB')
        
        fig, axes = plt.subplots(1, 6, figsize=(20, 5))
        fig.suptitle(f"Dataset: {dataset_name} | Image Index: {idx}", fontsize=16)
        
        # 0. Original Image
        axes[0].imshow(img_rgb)
        axes[0].set_title("Original")
        axes[0].axis('off')
        
        # 1-5. Model Heatmaps
        for i, model in enumerate(vision_models):
            npy_file = os.path.join(saliency_dir, f"Saliency_{model}_{idx}.npy")
            
            if os.path.exists(npy_file):
                heatmap = np.load(npy_file)
                # Plot heatmap with 'jet' colormap for high contrast
                im = axes[i+1].imshow(heatmap, cmap='jet')
                axes[i+1].set_title(model)
            else:
                axes[i+1].text(0.5, 0.5, "MISSING", ha='center')
                
            axes[i+1].axis('off')
            
        plt.tight_layout()
        plt.show()

# Run the check for the first 2 images of each dataset
for ds in ALL_DATASETS:
    print(f"\nVALIDATING: {ds}")
    validate_saliency_maps(ds, num_images=2)

In [ ]:
def get_word_category(token):
    """
    Categorizes tokens while isolating technical model tokens.
    """
    # Technical infrastructure tokens (Model Sinks)
    technical_tokens = {
        '[CLS]', '[SEP]', '<s>', '</s>', '<|endoftext|>', 
        '<pad>', '[PAD]', '<mask>', '<unk>'
    }
    
    # Standard English Stopwords
    stopwords = {
        'a', 'an', 'the', 'is', 'are', 'was', 'were', 
        'on', 'in', 'at', 'of', 'and', 'with', 'to', 'for'
    }
    
    # Clean BERT (##) and RoBERTa/GPT2 (Ġ) markers
    clean_token = token.replace('##', '').replace('Ġ', '').lower()
    
    if clean_token in technical_tokens:
        return "TECHNICAL"
    if clean_token in stopwords or not clean_token.isalnum():
        return "Grammar/Stopwords"
    return "Content (Nouns/Verbs)"

def calculate_semantic_density_refined(img_idx, dataset_name, text_models):
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    results = []

    for model_label in text_models:
        path = os.path.join(saliency_dir, f"SaliencyText_{model_label}_{img_idx}.npy")
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True).item()
            tokens, scores = data['tokens'], data['scores']
            
            content_sum = 0
            grammar_sum = 0
            
            for t, s in zip(tokens, scores):
                cat = get_word_category(t)
                if cat == "TECHNICAL":
                    continue # Exclude [CLS], [SEP], etc. from the percentage
                elif cat == "Content (Nouns/Verbs)":
                    content_sum += s
                else:
                    grammar_sum += s
            
            total = content_sum + grammar_sum
            if total > 0:
                results.append({
                    "Model": model_label.upper(),
                    "Content (%)": (content_sum / total) * 100,
                    "Grammar (%)": (grammar_sum / total) * 100
                })

    df_density = pd.DataFrame(results)
    
    # Visualization
    ax = df_density.set_index("Model").plot(
        kind='bar', stacked=True, figsize=(10, 6), color=['#2ecc71', '#e74c3c']
    )
    plt.title(f"Refined Semantic Density (Excluding Technical Tokens) | ID: {img_idx}", fontweight='bold')
    plt.ylabel("Percentage of Word-Level Attribution")
    plt.legend(title="Token Type", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    return df_density

In [ ]:
def aggregate_to_words(tokens, scores):
    """Aggregates sub-token scores into full word scores for direct comparison."""
    words = []
    word_scores = []
    current_word = ""
    current_score = 0
    count = 0

    for t, s in zip(tokens, scores):
        # Handle BERT (##) and RoBERTa/GPT2 (Ġ) sub-tokens
        is_subtoken = t.startswith("##") or (not t.startswith("Ġ") and count > 0 and "clip" not in t) 
        # Note: Logic varies by tokenizer, this is a generalized heuristic
        
        if is_subtoken:
            current_word += t.replace("##", "")
            current_score = max(current_score, s) # Take the max importance of sub-tokens
        else:
            if current_word:
                words.append(current_word.replace("Ġ", ""))
                word_scores.append(current_score)
            current_word = t
            current_score = s
        count += 1
    
    words.append(current_word.replace("Ġ", ""))
    word_scores.append(current_score)
    return words, word_scores

def calculate_model_correlation(img_idx, dataset_name, text_models):
    """
    Computes Spearman Correlation between the saliency rankings of different models.
    """
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    model_data = {}

    for model in text_models:
        path = os.path.join(saliency_dir, f"SaliencyText_{model}_{img_idx}.npy")
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True).item()
            # Aggregate to words so we compare the same units
            words, scores = aggregate_to_words(data['tokens'], data['scores'])
            model_data[model] = scores

    # Create correlation matrix
    names = list(model_data.keys())
    matrix = np.zeros((len(names), len(names)))

    for i, name1 in enumerate(names):
        for j, name2 in enumerate(names):
            # We compare ranks of words
            s1, s2 = model_data[name1], model_data[name2]
            # Ensure lengths match (truncate to shortest for comparison)
            min_len = min(len(s1), len(s2))
            corr, _ = spearmanr(s1[:min_len], s2[:min_len])
            matrix[i, j] = corr

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, xticklabels=[n.upper() for n in names], yticklabels=[n.upper() for n in names], cmap="coolwarm", center=0)
    plt.title(f"XAI Rank Correlation (Spearman) | Index: {img_idx}")
    plt.show()

In [ ]:
# 1. Define the loading function (Required for the loop)
def load_texts(dataset_name):
    df_path = os.path.join(DATASETS_DIR, f"df_{dataset_name}.pkl")
    if not os.path.exists(df_path):
        raise FileNotFoundError(f"Dataframe not found: {df_path}")
    df_temp = pd.read_pickle(df_path)
    # Get the first caption for each image index
    return [caps[0] if isinstance(caps, list) else str(caps) for caps in df_temp['captions']]

# 2. Your Execution Block (Modified for all 5 models)
text_models_xai = ['bert', 'roberta', 'gpt2', 'clip_text', 'rnn'] # All models
all_datasets = ['Flickr8k', 'Flickr30k', 'ConceptualCaptions']

for ds_name in all_datasets:
    print(f"\n{'='*20}\nProcessing Dataset: {ds_name}\n{'='*20}")
    
    dataset_texts = load_texts(ds_name)
    ds_dir = os.path.join(BASE_DIR, f"TextSaliency_{ds_name}")
    os.makedirs(ds_dir, exist_ok=True)
    
    for model_name in text_models_xai:
        print(f"\n--- Extracting {model_name.upper()} embeddings and saliency ---")
        
        # Use existing _xai functions for Transformers
        if model_name == 'bert':
            feats = get_bert_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'clip_text':
            feats = get_clip_text_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'rnn':
            feats = get_rnn_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'roberta':
            feats = get_roberta_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'gpt2':
            feats = get_gpt2_embeddings_xai(dataset_texts, device, ds_name)
        
        # ADDED: CLIP and RNN support (They need their own logic)
        
        
        # Save features to RAW_DIR for future fusion
        feat_save_path = os.path.join(RAW_DIR, f"X_text_{model_name}_raw_{ds_name}.npy")
        np.save(feat_save_path, feats)
        
        # Clear VRAM
        del feats
        torch.cuda.empty_cache()
        gc.collect()

    send_ntfy_notification(f"Text XAI completed for {ds_name}.", "TFE Pipeline Update")

In [ ]:
calculate_semantic_density_refined(
    img_idx=0,
    dataset_name="Flickr8k",
    text_models=["bert", "roberta", "gpt2", "clip_text"]
)

In [ ]:
calculate_model_correlation(
    img_idx=0,
    dataset_name="Flickr8k",
    text_models=["bert", "roberta", "gpt2", "clip_text"]
)